# Detecció d'exoplanetes amb dades del telescopi Kepler
Aprenentatge Automàtic · Grau en Estadística Aplicada (UAB)

Codi de l'annex de l'informe (`informe.pdf`). Classificació de senyals del telescopi Kepler
(exoplaneta confirmat o fals positiu) amb K-NN, SVM i Random Forest.

Dades: *Kepler Exoplanet Search Results* (NASA Exoplanet Archive), fitxer `cumulative.csv`.

In [ ]:
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy import stats

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, matthews_corrcoef

## Creació de la base de dades i estudi de valors nuls

In [ ]:
# Carreguem la base de dades
df = pd.read_csv('cumulative.csv')

# Seleccionem només les variables d'interès
vars_interes = ["kepoi_name", "koi_disposition", "koi_fpflag_nt", "koi_fpflag_ss",
                "koi_fpflag_co", "koi_fpflag_ec", "koi_period", "koi_impact",
                "koi_duration", "koi_depth", "koi_prad", "koi_teq", "koi_insol",
                "koi_steff", "koi_slogg", "koi_srad", "koi_kepmag"]
df = df[vars_interes]
df.head()
print(df.isnull().sum())
df = df.dropna()
len(df)

## Distribució de la variable resposta

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='koi_disposition', data=df, palette='viridis')
plt.title('Distribucio de les Classes (Target)')
plt.show()

## Estadístics descriptius

In [ ]:
df = df[df["koi_disposition"] != "CANDIDATE"]
desc_stats = df.describe().transpose()
display(desc_stats)

## Histogrames

In [ ]:
# Histogrames de variables clau (amb escala logarítmica per veure millor els atípics)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Radi del planeta
sns.histplot(df['koi_prad'].dropna(), bins=50, ax=axes[0], log_scale=True, color='teal')
axes[0].set_title('Distribucio del Radi Planetari')
axes[0].set_xlabel('Log (Radi) (Radis Terrestres)')
axes[0].set_ylabel('Nombre')

# Període orbital
sns.histplot(df['koi_period'].dropna(), bins=50, ax=axes[1], log_scale=True, color='orange')
axes[1].set_title('Distribucio del Periode Orbital')
axes[1].set_xlabel('Log (periode) (dies)')
axes[1].set_ylabel('Nombre')

# Profunditat del trànsit
sns.histplot(df['koi_depth'].dropna(), bins=50, ax=axes[2], log_scale=True, color='purple')
axes[2].set_title('Distribucio de la Profunditat del Transit')
axes[2].set_xlabel('Log (profunditat) (ppm)')
axes[2].set_ylabel('Nombre')
plt.show()

In [ ]:
# Més histogrames de variables predictores
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Durada del trànsit
sns.histplot(df['koi_duration'].dropna(), bins=50, ax=axes[0], color='teal')
axes[0].set_title('Distribucio duracio transit')
axes[0].set_xlabel('Duration (hores)')
axes[0].set_yscale('log')
axes[0].set_ylabel('Log (Nombre)')

# Temperatura de l'estrella
sns.histplot(df['koi_steff'].dropna(), bins=50, ax=axes[1], color='orange')
axes[1].set_title('Distribucio temperatura estrella')
axes[1].set_xlabel('Temperatura (Kelvin)')
axes[1].set_ylabel('Nombre')

# Estimació de la temperatura del planeta
sns.histplot(df['koi_teq'].dropna(), bins=50, ax=axes[2], color='purple')
axes[2].set_title('Distribucio temperatura planeta (estimacio)')
axes[2].set_xlabel('Temperatura (Kelvin)')
axes[2].set_ylabel('Nombre')
plt.show()

## Correlacions

In [ ]:
plt.figure(figsize=(12, 10))
# Seleccionem només les numèriques per a la matriu
numeric_df = df.select_dtypes(include=['float64', 'int64'])
sns.heatmap(numeric_df.corr(), cmap='coolwarm', annot=True)
plt.title('Matriu de Correlacio')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

# Diagrama de dispersió
sns.scatterplot(
    data=df,
    x='koi_impact',
    y='koi_prad',
    hue='koi_disposition',
    style='koi_disposition',
    palette='Set1',
    alpha=0.6
)

plt.title('Relacio entre Radi Planetari i Parametre d\'Impacte')
plt.xlabel('Parametre d\'Impacte (Unitats de Radi Estelar)')
plt.ylabel('Radi Planetari (Radis Terrestres)')

# Escala logarítmica per gestionar valors atípics
plt.xscale('log')
plt.yscale('log')
plt.grid(True, linestyle='--', alpha=0.5)

plt.legend(title='Disposicio del KOI')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

# Diagrama de dispersió
sns.scatterplot(
    data=df,
    x='koi_slogg',
    y='koi_srad',
    hue='koi_disposition',
    style='koi_disposition',
    palette='Set1',
    alpha=0.6
)

plt.title('Relacio entre Radi estelar (koi_srad) i Gravetat a la superficie de l\'estrella (koi_slogg)')
plt.xlabel('Gravetat (Log base 10 de l\'acceleracio)')
plt.ylabel('Radi Estelar (Radis Solars)')

plt.grid(True, linestyle='--', alpha=0.5)

plt.legend(title='Disposicio del KOI')
plt.show()

## Funció per al càlcul de mètriques

In [ ]:
def metriques(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob)
    }

## K-NN

In [ ]:
# Preparació de les dades
X = df.drop(columns=['kepoi_name', 'koi_disposition'])
y = df['koi_disposition'].apply(lambda x: 1 if x == 'CONFIRMED' else 0)

# Split train-test (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123, stratify=y
)

k_val = [1, 3, 5, 7, 10, 15, 20, 30]

results = {
    'k': [],
    'acc': [],
    'f1': [],
    'mcc': [],
    'auc': [],
}

# Normalitzar les dades
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Avaluació del model
for k in k_val:
    # Entrenar el model
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)

    # Predir
    y_pred = knn.predict(X_test_scaled)
    y_pred_proba = knn.predict_proba(X_test_scaled)[:, 1]

    # Calcular mètriques
    met = metriques(y_test, y_pred, y_pred_proba)

    # Desar resultats
    results['k'].append(k)
    results['acc'].append(met['Accuracy'])
    results['f1'].append(met['F1 Score'])
    results['mcc'].append(met['MCC'])
    results['auc'].append(met['AUC'])

results = pd.DataFrame(results)
print(results)

## SVM

In [ ]:
# Preparació de les dades
X = df.drop(columns=['kepoi_name', 'koi_disposition'])
y = df['koi_disposition'].apply(lambda x: 1 if x == 'CONFIRMED' else 0)

# Split train-test (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123, stratify=y
)

results = {
    'Config': [],
    'kernel': [],
    'C': [],
    'gamma': [],
    'acc': [],
    'f1': [],
    'mcc': [],
    'auc': [],
}

# Normalitzar les dades
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_test)

configs_svm = [
    {'C': 1, 'kernel': 'linear'},
    {'C': 10, 'kernel': 'linear'},
    {'C': 1, 'kernel': 'rbf', 'gamma': 'scale'},
    {'C': 10, 'kernel': 'rbf', 'gamma': 'scale'},
    {'C': 1, 'kernel': 'rbf', 'gamma': 0.1},
    {'C': 10, 'kernel': 'rbf', 'gamma': 0.1},
]

# Avaluació del model
for config in configs_svm:
    svm = SVC(**config, random_state=123, probability=True)
    svm.fit(X_train_scaled, y_train)

    y_pred_val = svm.predict(X_val_scaled)
    y_pred_proba_val = svm.predict_proba(X_val_scaled)[:, 1]

    met = metriques(y_test, y_pred_val, y_pred_proba_val)

    results['Config'].append(f"C{config['C']}_{config['kernel']}_g:{config.get('gamma', 'N/A')}")
    results['kernel'].append(config['kernel'])
    results['C'].append(config['C'])
    results['gamma'].append(config.get('gamma', 'N/A'))
    results['acc'].append(round(met['Accuracy'], 4))
    results['f1'].append(round(met['F1 Score'], 4))
    results['mcc'].append(round(met['MCC'], 4))
    results['auc'].append(round(met['AUC'], 4))

# Convertir a DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='auc', ascending=False)
print(results_df.to_string(index=False))

## Random Forest

In [ ]:
X = df.drop(columns=['kepoi_name', 'koi_disposition'])
y = df['koi_disposition'].apply(lambda x: 1 if x == 'CONFIRMED' else 0)

# Split train-test (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123, stratify=y
)

nl = [50, 100, 200, 300]
max_depth = [None, 10, 20, 30]
results = []

for n in nl:
    for md in max_depth:
        rf = RandomForestClassifier(
            n_estimators=n,
            max_depth=md,
            random_state=123,
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_test)
        y_pred_proba = rf.predict_proba(X_test)[:, 1]

        met = metriques(y_test, y_pred, y_pred_proba)

        results.append({
            'n_estimators': n,
            'max_depth': md,
            'accuracy': round(met['Accuracy'], 4),
            'f1': round(met['F1 Score'], 4),
            'mcc': round(met['MCC'], 4),
            'auc': round(met['AUC'], 4)
        })

results_df = pd.DataFrame(results)

# Ordenar per AUC
results_df_sorted = results_df.sort_values('auc', ascending=False)
print(results_df_sorted)

# Millor configuració
best_config = results_df_sorted.iloc[0]
print("\nMillor configuracio:")
print(best_config)

## Importància de les variables

In [ ]:
best_rf = RandomForestClassifier(n_estimators=100, max_depth=30, random_state=123)
best_rf.fit(X_train, y_train)

importances = best_rf.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Importancia de les Variables - Random Forest')
plt.xlabel('Importancia relativa')
plt.ylabel('Variable (KOI parameter)')
plt.show()

## Validació creuada (10 folds) i comparació estadística dels models

In [ ]:
models = {
    'K-NN': Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=7))]),
    'SVM': Pipeline([('scaler', StandardScaler()), ('svm', SVC(kernel='rbf', C=10, gamma=0.1, probability=True))]),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=30, random_state=123)
}

# Configuració de la validació creuada (10 folds)
kf = KFold(n_splits=10, shuffle=True, random_state=123)

# Obtenció de les accuracy
cv_scores = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')
    cv_scores[name] = scores

results = []
comparacions = list(itertools.combinations(models.keys(), 2))

for m1, m2 in comparacions:
    # Càlcul de la diferència
    diff = cv_scores[m1] - cv_scores[m2]

    # Test de normalitat (Shapiro-Wilk) sobre les diferències
    shapiro_p = stats.shapiro(diff).pvalue

    # Elecció del test segons la normalitat (alfa = 0.05)
    if shapiro_p > 0.05:
        test_name = "t-test"
        t_stat, p_val = stats.ttest_rel(cv_scores[m1], cv_scores[m2])
    else:
        test_name = "Wilcoxon"
        stat, p_val = stats.wilcoxon(cv_scores[m1], cv_scores[m2])

    results.append({
        'Comparacio': f"{m1} vs {m2}",
        'Shapiro p': round(shapiro_p, 4),
        'Test': test_name,
        'p-valor Sig.': round(p_val, 4),
    })

df_stats = pd.DataFrame(results)
print(df_stats)